# Index-Enhanced Fund Analysis (Wind)

Benchmarks the 中证500/800/1000/2000 and 国证2000 active-quant / index-enhanced funds (AUM > 2亿, inception > 1 year) against their respective benchmark indices.

## Setup: connect to WindPy

In [ ]:
import os ;\
import site ;\
import sys ;\
os.system('mkdir -p ' + site.getusersitepackages()) ;\
os.system('ln -sf "/Applications/Wind API.app/Contents/python/WindPy.py"' + ' ' + site.getusersitepackages()) ;\
os.system('ln -sf "/Applications/Wind API.app/Contents/python/WindPy.py"' + ' ' + site.getsitepackages()[0]) ;\
os.system('rm -rf ~/.Wind') ;\
os.system('ln -sf ~/Library/Containers/com.wind.mac.api/Data/.Wind ~/.Wind') ;\
print("Current Python Version: " + sys.version) ;\
print("Current Python Env: " + sys.executable) ;\
print("WindPy installed at: " + site.getusersitepackages()) ;\
print("WindPy installed at: " + site.getsitepackages()[0])

In [ ]:
from WindPy import *

ret = w.start()
print(ret)

ret = w.isconnected()
print(ret)

#test WSS function
ret = w.wss("000001.SZ", "sec_name","")
print(ret)

## Imports

In [ ]:
import os
import pandas as pd
import numpy as np


## Fund universe: codes by benchmark group

In [ ]:
# 中证500/800/1000/2000 国证1000/2000为基准的主动量化和指数增强基金（aum大于2亿，成立时间大于一年）

# fund code for 中证500 
fund_codes_SH500 = [
    "006729.OF", "006730.OF", "014305.OF", "014306.OF", "004945.OF", "561550.OF", "001556.OF", "001557.OF", "159610.OF",
    "006682.OF", "016935.OF", "012080.OF", "012081.OF", "002316.OF", "006048.OF", "007413.OF", "003016.OF", "014344.OF", "014155.OF", "003578.OF",
    "003986.OF", "014156.OF", "006593.OF", "006594.OF", "502000.OF", "009300.OF", "015508.OF", "005994.OF", "002906.OF", "007089.OF", "002907.OF", "005062.OF",
    "000478.OF", "161017.OF", "004192.OF", "013332.OF", "004193.OF", "001050.OF", "016854.OF", "021186.OF"
]

# fund code for 中证800
fund_codes_SH800 = ["016276.OF", "022513.OF", "010673.OF"]

# fund code for 中证1000
fund_codes_SH1000 = [
    "018013.OF", "018014.OF", "015466.OF", "017094.OF", "017095.OF", "017953.OF", "017954.OF", "019240.OF", "019241.OF", "005313.OF",
    "017644.OF", "005314.OF", "017645.OF", "017846.OF", "006165.OF", "015867.OF", "017847.OF", "006166.OF", "015868.OF", "019555.OF", "014201.OF",
    "014202.OF", "018157.OF", "018158.OF", "159680.OF", "161039.OF", "013331.OF", "017919.OF", "017920.OF", "016936.OF", "016937.OF", "016785.OF", "016942.OF", "016943.OF",
    "004194.OF", "004195.OF", "015784.OF"]
    
# fund code for 中证2000
fund_codes_SH2000 = ["019918.OF", "019919.OF", "159552.OF"]

# no fund for 国证1000 

# fund code for 国证2000
fund_codes_GZ2000 = ["019318.OF", "019319.OF", "018653.OF"]

In [ ]:
# 主动量化基金 (active quant, not index-enhanced) benchmarked against 中证500
# NOTE: 012080.OF and 005994.OF also appear in fund_codes_SH500 above (both are
# named as index-enhanced funds) - flagging in case that's a categorization mix-up.
fund_codes_ACTIVE_QUANT_SH500 = [
    "007808.OF",
    "010779.OF",
    "006195.OF",
    "014805.OF",
    "001421.OF",
    "004641.OF",
    "004695.OF",
    "014135.OF",
    "001917.OF",
    "001244.OF",
    "013166.OF",
    "001990.OF",
    "006648.OF",
    "007831.OF",
    "011934.OF",
    "460009.OF",
    "021692.OF",
    "166107.OF",
    "012080.OF",
    "000006.OF",
    "001637.OF",
    "022909.OF",
    "005994.OF",
    "000978.OF",
    "014691.OF",
    "017715.OF",
    "161038.OF",
    "016267.OF",
    "004250.OF",
    "233009.OF",
    "014187.OF",
    "003717.OF",
    "006336.OF",
    "163110.OF",
    "011824.OF",
    "005120.OF",
    "011410.OF",
    "019561.OF",
    "010120.OF",
    "005189.OF",
    "005126.OF",
    "012977.OF",
    "020117.OF",
    "009043.OF",
    "002270.OF",
    "003241.OF",
    "018705.OF",
    "021404.OF",
    "005865.OF",
    "009874.OF",
    "011589.OF",
    "025984.OF",
    "025982.OF",
    "025386.OF",
    "025011.OF",
    "025013.OF",
    "025466.OF",
    "025447.OF"
]


## Data fetching functions

In [ ]:
def get_index_daily_returns(index_code, start_date, end_date, save_path=None, force_refresh=False):
    if save_path and not force_refresh and os.path.exists(save_path):
        print(f"Loading cached index data from {save_path} (skip Wind fetch to save API quota)")
        return pd.read_csv(save_path, index_col=0)

    error_code, data = w.wsd(index_code, "pct_chg", start_date, end_date, "", usedf=True)

    if error_code != 0:
        print(error_code)
        print(data)
        return None

    if save_path:
        data.to_csv(save_path, index=True)

    return data


In [ ]:
def get_fund_nav_data(fund_codes, start_date, end_date, save_path=None, force_refresh=False):
      if save_path and not force_refresh and os.path.exists(save_path):
            print(f"Loading cached NAV data from {save_path} (skip Wind fetch to save API quota)")
            return pd.read_csv(save_path, index_col=0)

      w.start()

      error_code, nav_data = w.wsd(
            fund_codes,
            "NAV_acc",  # cumulative NAV: avoids artificial drops on fund distribution dates
            start_date,
            end_date,
            "",
            usedf=True,
      )

      name_error_code, name_data = w.wss(
            fund_codes,
            "sec_name",
            usedf=True,
      )

      if error_code != 0:
            print(error_code)
            print(nav_data)
            return None

      if name_error_code != 0:
            print(name_error_code)
            print(name_data)
            return None

      chart = nav_data.T
      chart.index.name = "fund_code"
      chart.columns = [pd.Timestamp(col).strftime("%Y-%m-%d") for col in chart.columns]

      fund_names = name_data.iloc[:, 0].reindex(chart.index)
      chart = pd.concat([fund_names.rename("fund_name"), chart], axis=1)

      if save_path:
            chart.to_csv(save_path, index=True)

      return chart


def calculate_fund_daily_returns(fund_nav_data, save_path=None):
      if isinstance(fund_nav_data, str):
            fund_nav_data = pd.read_csv(fund_nav_data, index_col=0)
      else:
            fund_nav_data = fund_nav_data.copy()

      fund_names = fund_nav_data["fund_name"]
      nav_data = fund_nav_data.drop(columns=["fund_name"]).copy()
      # Drop only the first date column, which pct_change always makes NaN for
      # every fund (no prior day to diff against). Do NOT use dropna(axis=1) here -
      # that would drop any date where even one fund lacks data (e.g. a fund that
      # launched partway through the window), truncating the whole group's date
      # range to only the period after the last fund to launch. Individual funds
      # simply carry NaN before their own inception, which downstream mean/std
      # calls already handle correctly per-fund (skipna is the pandas default).
      calculated_returns = nav_data.pct_change(axis=1).iloc[:, 1:] * 100
      calculated_returns = pd.concat(
            [fund_names.reindex(calculated_returns.index).rename("fund_name"), calculated_returns],
            axis=1,
      )

      if save_path:
            calculated_returns.to_csv(save_path, index=True)

      return calculated_returns


## Generalize across all benchmark groups + risk/ranking metrics

In [ ]:
# Map each benchmark group to its fund codes and index code.
#
# NOTE: the original `index_codes` list had 000852.SH (CSI1000) listed twice -
# once for SH1000 and once for SH2000 - which looks like a copy/paste bug.

FUND_GROUPS = {
    "SH500":  {"fund_codes": fund_codes_SH500,  "index_code": "000905.SH"},
    "SH800":  {"fund_codes": fund_codes_SH800,  "index_code": "000906.SH"},
    "SH1000": {"fund_codes": fund_codes_SH1000, "index_code": "000852.SH"},
    "SH2000": {"fund_codes": fund_codes_SH2000, "index_code": "932000.CSI"},
    "CN2000": {"fund_codes": fund_codes_GZ2000, "index_code": "399303.SZ"},
    "SH500_ActiveQuant": {"fund_codes": fund_codes_ACTIVE_QUANT_SH500, "index_code": "000905.SH"},
}


In [ ]:
def calculate_excess_returns(fund_daily_returns, benchmark_daily_returns, save_path=None):
    """Daily excess return (%) = fund daily return - benchmark daily return, per fund."""
    fund_names = fund_daily_returns["fund_name"]
    returns = fund_daily_returns.drop(columns=["fund_name"])

    benchmark = benchmark_daily_returns.copy()
    benchmark.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in benchmark.index]
    benchmark_row = benchmark.iloc[:, 0].reindex(returns.columns)

    excess = returns.sub(benchmark_row, axis=1)
    excess = pd.concat([fund_names.reindex(excess.index), excess], axis=1)

    if save_path:
        excess.to_csv(save_path, index=True)

    return excess


def calculate_max_drawdown(fund_nav_data, save_path=None):
    """Max drawdown (%) per fund, computed from NAV levels (not from returns)."""
    fund_names = fund_nav_data["fund_name"]
    nav = fund_nav_data.drop(columns=["fund_name"]).astype(float)

    running_max = nav.cummax(axis=1)
    drawdown = (nav - running_max) / running_max * 100
    max_dd = drawdown.min(axis=1)

    result = pd.DataFrame({"fund_name": fund_names, "max_drawdown_pct": max_dd})

    if save_path:
        result.to_csv(save_path, index=True)

    return result


def build_fund_scorecard(fund_nav_data, fund_daily_returns, benchmark_daily_returns,
                          trading_days=242, save_path=None):
    """
    One row per fund: annualized return/vol/Sharpe, annualized excess return,
    tracking error, and information ratio vs. the group benchmark, plus max drawdown.
    Sorted by information ratio (descending).
    """
    fund_names = fund_daily_returns["fund_name"]
    returns = fund_daily_returns.drop(columns=["fund_name"]) / 100  # back to decimal

    excess_returns_pct = calculate_excess_returns(fund_daily_returns, benchmark_daily_returns)
    excess = excess_returns_pct.drop(columns=["fund_name"]) / 100

    max_dd = calculate_max_drawdown(fund_nav_data)

    ann_return = returns.mean(axis=1) * trading_days * 100
    ann_vol = returns.std(axis=1) * np.sqrt(trading_days) * 100
    sharpe = ann_return / ann_vol

    ann_excess_return = excess.mean(axis=1) * trading_days * 100
    tracking_error = excess.std(axis=1) * np.sqrt(trading_days) * 100
    information_ratio = ann_excess_return / tracking_error

    scorecard = pd.DataFrame({
        "fund_name": fund_names,
        "ann_return_pct": ann_return,
        "ann_vol_pct": ann_vol,
        "sharpe": sharpe,
        "ann_excess_return_pct": ann_excess_return,
        "tracking_error_pct": tracking_error,
        "information_ratio": information_ratio,
    }).join(max_dd["max_drawdown_pct"])

    scorecard = scorecard.sort_values("information_ratio", ascending=False)

    if save_path:
        scorecard.to_csv(save_path, index=True)

    return scorecard

In [ ]:
# Run the full pipeline (NAV -> daily returns -> benchmark -> scorecard) for every group.
scorecards = {}
nav_data_by_group = {}
daily_returns_by_group = {}
benchmark_returns_by_group = {}
benchmark_returns_by_index_code = {}  # groups sharing an index (e.g. SH500 and
                                       # SH500_ActiveQuant both use 000905.SH) only fetch it once

for group_name, group in FUND_GROUPS.items():
    fund_codes = group["fund_codes"]
    index_code = group["index_code"]

    if not fund_codes:
        continue

    nav_data = get_fund_nav_data(
        fund_codes, "2025-07-01", "2026-07-01",
        save_path=f"fund_nav_data_{group_name.lower()}.csv",
    )
    if nav_data is None:
        print(f"Skipping {group_name}: failed to fetch fund NAV data (see Wind error above).")
        continue

    if index_code in benchmark_returns_by_index_code:
        benchmark_returns = benchmark_returns_by_index_code[index_code]
    else:
        benchmark_returns = get_index_daily_returns(
            index_code, "2025-07-01", "2026-07-01",
            save_path=f"index_daily_returns_{group_name.lower()}.csv",
        )
        if benchmark_returns is not None:
            benchmark_returns_by_index_code[index_code] = benchmark_returns

    if benchmark_returns is None:
        print(f"Skipping {group_name}: failed to fetch benchmark index data (see Wind error above).")
        continue

    daily_returns = calculate_fund_daily_returns(
        nav_data, save_path=f"fund_daily_returns_{group_name.lower()}.csv",
    )

    scorecard = build_fund_scorecard(
        nav_data, daily_returns, benchmark_returns,
        save_path=f"fund_scorecard_{group_name.lower()}.csv",
    )
    scorecard.insert(0, "benchmark_group", group_name)
    scorecards[group_name] = scorecard

    # Keep the underlying series around (not just the scorecard) for the
    # cross-group and top-performer deep-dive sections below.
    nav_data_by_group[group_name] = nav_data
    daily_returns_by_group[group_name] = daily_returns
    benchmark_returns_by_group[group_name] = benchmark_returns

master_scorecard = pd.concat(scorecards.values(), axis=0)
master_scorecard.to_csv("fund_scorecard_all.csv", index=True)
print(master_scorecard)


## Visualize the scorecard

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import font_manager
import os

# Pick the first installed font that supports CJK characters, so fund names
# (in Chinese) render instead of showing up as missing-glyph boxes.
_CJK_FONT_CANDIDATES = [
    "PingFang SC", "Heiti SC", "STHeiti", "Arial Unicode MS",
    "Microsoft YaHei", "SimHei", "Noto Sans CJK SC", "WenQuanYi Zen Hei",
]
_available_fonts = {f.name for f in font_manager.fontManager.ttflist}
_cjk_font = next((f for f in _CJK_FONT_CANDIDATES if f in _available_fonts), None)

# Fallback: macOS/Windows CJK fonts often ship as .ttc collection files that
# matplotlib's font cache doesn't always index by family name, so the name-based
# lookup above can miss a font that is actually installed. Register known font
# file paths directly instead.
if _cjk_font is None:
    _CJK_FONT_PATHS = [
        "/System/Library/Fonts/PingFang.ttc",
        "/System/Library/Fonts/STHeiti Light.ttc",
        "/System/Library/Fonts/STHeiti Medium.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
        "C:/Windows/Fonts/msyh.ttc",
        "C:/Windows/Fonts/simhei.ttf",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
    ]
    for _path in _CJK_FONT_PATHS:
        if os.path.exists(_path):
            font_manager.fontManager.addfont(_path)
            _cjk_font = font_manager.FontProperties(fname=_path).get_name()
            break

if _cjk_font:
    plt.rcParams["font.sans-serif"] = [_cjk_font]
    plt.rcParams["font.family"] = "sans-serif"
else:
    print(
        "No CJK font found by name or file path - Chinese fund names will still "
        "show as boxes. Install a CJK font (e.g. Noto Sans CJK) and rerun this cell."
    )

plt.rcParams["axes.unicode_minus"] = False  # avoid garbled minus signs with CJK fonts
print("Using font:", _cjk_font)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

groups = master_scorecard["benchmark_group"].unique()
colors = plt.cm.tab10.colors

for i, group in enumerate(groups):
    subset = master_scorecard[master_scorecard["benchmark_group"] == group]
    ax.scatter(
        subset["tracking_error_pct"],
        subset["information_ratio"],
        s=subset["ann_vol_pct"] * 3,  # bubble size ~ annualized volatility
        alpha=0.7,
        color=colors[i % len(colors)],
        label=group,
    )

ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax.set_xlabel("Tracking Error (%)")
ax.set_ylabel("Information Ratio")
ax.set_title("Information Ratio vs. Tracking Error (bubble size = annualized volatility)")
ax.legend(title="Benchmark Group")

plt.tight_layout()
plt.savefig("fund_ir_vs_te_scatter.png", dpi=150)
plt.show()

In [ ]:
top15 = master_scorecard.sort_values("information_ratio", ascending=False).head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top15["fund_name"], top15["information_ratio"], color="steelblue")
ax.invert_yaxis()
ax.set_xlabel("Information Ratio")
ax.set_title("Top 15 Funds by Information Ratio (across all benchmark groups)")
plt.tight_layout()
plt.savefig("fund_top15_ir_bar.png", dpi=150)
plt.show()

## Cross-benchmark-group comparison

In [ ]:
group_summary = master_scorecard.groupby("benchmark_group")[
    ["ann_return_pct", "ann_vol_pct", "sharpe", "ann_excess_return_pct",
     "tracking_error_pct", "information_ratio", "max_drawdown_pct"]
].agg(["mean", "median", "std"])

print(group_summary)


In [ ]:
group_order = [g for g in FUND_GROUPS if g in daily_returns_by_group]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ir_by_group = [
    master_scorecard.loc[master_scorecard["benchmark_group"] == g, "information_ratio"].dropna()
    for g in group_order
]
excess_by_group = [
    master_scorecard.loc[master_scorecard["benchmark_group"] == g, "ann_excess_return_pct"].dropna()
    for g in group_order
]

axes[0].boxplot(ir_by_group, labels=group_order)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_title("Information Ratio Distribution by Benchmark Group")
axes[0].set_ylabel("Information Ratio")

axes[1].boxplot(excess_by_group, labels=group_order)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[1].set_title("Annualized Excess Return Distribution by Benchmark Group")
axes[1].set_ylabel("Annualized Excess Return (%)")

plt.tight_layout()
plt.savefig("group_comparison_boxplots.png", dpi=150)
plt.show()


## Top-performer deep dive

In [ ]:
def cumulative_return_series(fund_code, group_name):
    """Cumulative return (indexed to 100) for a fund and its group benchmark, aligned by date."""
    daily_returns = daily_returns_by_group[group_name]
    benchmark_returns = benchmark_returns_by_group[group_name]

    fund_name = daily_returns.loc[fund_code, "fund_name"]
    fund_ret = daily_returns.loc[fund_code].drop("fund_name").astype(float) / 100

    benchmark = benchmark_returns.copy()
    benchmark.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in benchmark.index]
    bench_ret = benchmark.iloc[:, 0].reindex(fund_ret.index) / 100

    fund_cum = (1 + fund_ret).cumprod() * 100
    bench_cum = (1 + bench_ret).cumprod() * 100

    return fund_name, fund_cum, bench_cum


top5 = master_scorecard.sort_values("information_ratio", ascending=False).head(5)

fig, axes = plt.subplots(len(top5), 1, figsize=(10, 4 * len(top5)))

for ax, (fund_code, row) in zip(axes, top5.iterrows()):
    fund_name, fund_cum, bench_cum = cumulative_return_series(fund_code, row["benchmark_group"])
    x = pd.to_datetime(fund_cum.index, format="%Y-%m-%d")

    ax.plot(x, fund_cum.values, label=f"{fund_name} ({fund_code})")
    ax.plot(x, bench_cum.values, label=f"{row['benchmark_group']} benchmark", linestyle="--", color="gray")
    ax.set_title(f"{fund_name} ({fund_code}) vs {row['benchmark_group']} benchmark - IR={row['information_ratio']:.2f}")
    ax.set_ylabel("Cumulative Return (indexed to 100)")
    ax.legend()

plt.tight_layout()
plt.savefig("top5_cumulative_vs_benchmark.png", dpi=150)
plt.show()


In [ ]:
def flag_outlier_days(fund_code, group_name, z_thresh=3):
    """Fund daily returns more than z_thresh standard deviations from the fund's own mean."""
    daily_returns = daily_returns_by_group[group_name]
    ret = daily_returns.loc[fund_code].drop("fund_name").astype(float)
    z = (ret - ret.mean()) / ret.std()
    return ret[z.abs() > z_thresh]


for fund_code, row in top5.iterrows():
    outliers = flag_outlier_days(fund_code, row["benchmark_group"])
    if len(outliers):
        print(f"{row['fund_name']} ({fund_code}) - outlier days (>3 sigma from its own mean):")
        print(outliers)
        print()


In [ ]:
def split_period_ir(daily_returns, benchmark_daily_returns, trading_days_per_year=242):
    """
    Information ratio computed separately for the first and second half of the
    period. A fund with a strong IR in only one half is more likely a lucky
    stretch than a fund whose IR holds up in both halves.
    """
    fund_names = daily_returns["fund_name"]
    excess_pct = calculate_excess_returns(daily_returns, benchmark_daily_returns)
    excess = excess_pct.drop(columns=["fund_name"]) / 100

    mid = excess.shape[1] // 2
    halves = {"H1": excess.iloc[:, :mid], "H2": excess.iloc[:, mid:]}

    result = pd.DataFrame({"fund_name": fund_names})
    for label, sub in halves.items():
        ann_excess = sub.mean(axis=1) * trading_days_per_year * 100
        tracking_error = sub.std(axis=1) * np.sqrt(trading_days_per_year) * 100
        result[f"ir_{label}"] = ann_excess / tracking_error

    result["ir_persistence"] = result[["ir_H1", "ir_H2"]].min(axis=1)
    return result


persistence_frames = []
for group_name in daily_returns_by_group:
    p = split_period_ir(daily_returns_by_group[group_name], benchmark_returns_by_group[group_name])
    p.insert(0, "benchmark_group", group_name)
    persistence_frames.append(p)

persistence_table = pd.concat(persistence_frames, axis=0).sort_values("ir_persistence", ascending=False)
persistence_table.to_csv("fund_ir_persistence.csv", index=True)
print(persistence_table.head(15))


## Worst-performer deep dive

In [ ]:
bottom5 = master_scorecard.sort_values("information_ratio", ascending=True).head(5)

fig, axes = plt.subplots(len(bottom5), 1, figsize=(10, 4 * len(bottom5)))

for ax, (fund_code, row) in zip(axes, bottom5.iterrows()):
    fund_name, fund_cum, bench_cum = cumulative_return_series(fund_code, row["benchmark_group"])
    x = pd.to_datetime(fund_cum.index, format="%Y-%m-%d")

    ax.plot(x, fund_cum.values, label=f"{fund_name} ({fund_code})", color="firebrick")
    ax.plot(x, bench_cum.values, label=f"{row['benchmark_group']} benchmark", linestyle="--", color="gray")
    ax.set_title(f"{fund_name} ({fund_code}) vs {row['benchmark_group']} benchmark - IR={row['information_ratio']:.2f}")
    ax.set_ylabel("Cumulative Return (indexed to 100)")
    ax.legend()

plt.tight_layout()
plt.savefig("bottom5_cumulative_vs_benchmark.png", dpi=150)
plt.show()


In [ ]:
for fund_code, row in bottom5.iterrows():
    outliers = flag_outlier_days(fund_code, row["benchmark_group"])
    if len(outliers):
        print(f"{row['fund_name']} ({fund_code}) - outlier days (>3 sigma from its own mean):")
        print(outliers)
        print()


## Within-group ranking

In [ ]:
# Rank and percentile within each fund's own benchmark group - comparing IR
# straight across groups isn't fully fair, since some benchmarks are
# structurally easier to generate alpha against than others (see the
# cross-group comparison above).
master_scorecard["rank_in_group"] = (
    master_scorecard.groupby("benchmark_group")["information_ratio"]
    .rank(ascending=False, method="min")
    .astype(int)
)
master_scorecard["group_size"] = master_scorecard.groupby("benchmark_group")["information_ratio"].transform("size")
master_scorecard["percentile_in_group"] = 1 - (master_scorecard["rank_in_group"] - 1) / (
    (master_scorecard["group_size"] - 1).clip(lower=1)
)

master_scorecard.to_csv("fund_scorecard_all.csv", index=True)

display_cols = ["fund_name", "information_ratio", "ann_excess_return_pct", "max_drawdown_pct", "rank_in_group"]

for group_name in FUND_GROUPS:
    if group_name not in daily_returns_by_group:
        continue
    group_df = master_scorecard[master_scorecard["benchmark_group"] == group_name].sort_values("rank_in_group")
    print(f"=== {group_name}: best 3 (of {group_df.shape[0]}) ===")
    print(group_df.head(3)[display_cols])
    print(f"=== {group_name}: worst 3 ===")
    print(group_df.tail(3)[display_cols])
    print()


## Risk vs. reward: is extra tracking error actually rewarded?

In [ ]:
risk_reward_corr = (
    master_scorecard.groupby("benchmark_group")
    .apply(lambda df: df["tracking_error_pct"].corr(df["ann_excess_return_pct"]))
    .rename("corr_tracking_error_vs_excess_return")
)

print(risk_reward_corr)


In [ ]:
fig, axes = plt.subplots(1, len(group_order), figsize=(5 * len(group_order), 4), sharey=True)

for ax, group_name in zip(axes, group_order):
    subset = master_scorecard[master_scorecard["benchmark_group"] == group_name]
    ax.scatter(subset["tracking_error_pct"], subset["ann_excess_return_pct"], alpha=0.7)

    if len(subset) >= 2:
        coeffs = np.polyfit(subset["tracking_error_pct"], subset["ann_excess_return_pct"], 1)
        trend_x = np.linspace(subset["tracking_error_pct"].min(), subset["tracking_error_pct"].max(), 10)
        ax.plot(trend_x, np.polyval(coeffs, trend_x), color="firebrick", linestyle="--")

    ax.axhline(0, color="gray", linewidth=0.8, linestyle=":")
    corr = subset["tracking_error_pct"].corr(subset["ann_excess_return_pct"])
    ax.set_title(f"{group_name} (corr={corr:.2f})")
    ax.set_xlabel("Tracking Error (%)")

axes[0].set_ylabel("Annualized Excess Return (%)")
plt.tight_layout()
plt.savefig("risk_vs_reward_by_group.png", dpi=150)
plt.show()


## Active-quant (主动量化) fund group

58 actively-managed quant funds benchmarked against CSI500 (000905.SH), distinct from the index-enhanced SH500 group above.

In [ ]:
active_quant_group = master_scorecard[master_scorecard["benchmark_group"] == "SH500_ActiveQuant"]

# Fund code and name for every active-quant fund in this group.
print(active_quant_group[["fund_name"]])


In [ ]:
pd.set_option("display.max_rows", None)

active_quant_ranked = active_quant_group.sort_values("information_ratio", ascending=False)
print(active_quant_ranked[[
    "fund_name", "ann_return_pct", "ann_vol_pct", "sharpe",
    "ann_excess_return_pct", "tracking_error_pct", "information_ratio", "max_drawdown_pct",
]])

pd.reset_option("display.max_rows")


## Active-quant fund clustering: 58 strategies, or fewer crowded trades?

Unlike the index-enhanced funds (which must hold ≥80% index constituents), active-quant funds have much more freedom, so dispersion in what they actually do should be higher - this checks whether that freedom translates into genuinely different strategies or just different names on similar bets.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

def build_excess_return_matrix(group_name):
    """(funds x dates) daily excess return matrix for every fund in a group."""
    daily_returns = daily_returns_by_group[group_name]
    benchmark_returns = benchmark_returns_by_group[group_name]
    excess = calculate_excess_returns(daily_returns, benchmark_returns)
    return excess.drop(columns=["fund_name"])


quant_excess = build_excess_return_matrix("SH500_ActiveQuant")
quant_fund_names = daily_returns_by_group["SH500_ActiveQuant"]["fund_name"]

# Correlation of each fund's excess-return series with every other fund's -
# high correlation means two funds are effectively making the same bets.
quant_corr = quant_excess.T.corr()
print(quant_corr.shape)


In [ ]:
# Convert correlation to a distance (0 = identical, up to 2 = perfectly
# anti-correlated) and cluster funds that move together.
distance = 1 - quant_corr
np.fill_diagonal(distance.values, 0)  # exact zero on the diagonal, avoids float noise
condensed = squareform(distance.values, checks=False)
Z = linkage(condensed, method="average")

fig, ax = plt.subplots(figsize=(14, 6))
labels = [f"{code} {quant_fund_names[code]}" for code in quant_corr.index]
dendro = dendrogram(Z, labels=labels, leaf_rotation=90, leaf_font_size=7, ax=ax)
ax.set_title("Active-quant SH500 funds - clustered by excess-return correlation")
ax.set_ylabel("Distance (1 - correlation)")
plt.tight_layout()
plt.savefig("active_quant_dendrogram.png", dpi=150)
plt.show()


In [ ]:
# Reorder the correlation matrix to match the dendrogram's leaf order, so
# similar funds sit next to each other instead of in their original code order.
order = dendro["leaves"]
ordered_codes = [quant_corr.index[i] for i in order]
ordered_corr = quant_corr.loc[ordered_codes, ordered_codes]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(ordered_corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(ordered_codes)))
ax.set_yticks(range(len(ordered_codes)))
ax.set_xticklabels(ordered_codes, rotation=90, fontsize=6)
ax.set_yticklabels(ordered_codes, fontsize=6)
fig.colorbar(im, ax=ax, label="Correlation of excess return")
ax.set_title("Active-quant SH500 funds - excess-return correlation (clustered order)")
plt.tight_layout()
plt.savefig("active_quant_correlation_heatmap.png", dpi=150)
plt.show()


In [ ]:
# Cut the dendrogram into a fixed number of clusters (adjust n_clusters to taste)
# and see whether some "styles" are actually better than others.
n_clusters = 4
cluster_labels = fcluster(Z, t=n_clusters, criterion="maxclust")

quant_scorecard = master_scorecard[master_scorecard["benchmark_group"] == "SH500_ActiveQuant"]

cluster_df = pd.DataFrame({
    "fund_name": [quant_fund_names[c] for c in quant_corr.index],
    "cluster": cluster_labels,
}, index=quant_corr.index)
cluster_df = cluster_df.join(quant_scorecard[["information_ratio", "ann_excess_return_pct", "max_drawdown_pct"]])

for cluster_id in sorted(cluster_df["cluster"].unique()):
    members = cluster_df[cluster_df["cluster"] == cluster_id]
    print(f"=== Cluster {cluster_id} ({len(members)} funds) ===")
    print(members[["fund_name", "information_ratio", "ann_excess_return_pct", "max_drawdown_pct"]])
    print(
        f"cluster avg IR: {members['information_ratio'].mean():.2f}, "
        f"avg excess return: {members['ann_excess_return_pct'].mean():.2f}%\n"
    )


## Active-quant top 5 (within-group) vs CSI500

In [ ]:
quant_top5 = active_quant_group.sort_values("information_ratio", ascending=False).head(5)

fig, axes = plt.subplots(len(quant_top5), 1, figsize=(10, 4 * len(quant_top5)))

for ax, (fund_code, row) in zip(axes, quant_top5.iterrows()):
    fund_name, fund_cum, bench_cum = cumulative_return_series(fund_code, row["benchmark_group"])
    x = pd.to_datetime(fund_cum.index, format="%Y-%m-%d")

    ax.plot(x, fund_cum.values, label=f"{fund_name} ({fund_code})")
    ax.plot(x, bench_cum.values, label="CSI500 benchmark", linestyle="--", color="gray")
    ax.set_title(f"{fund_name} ({fund_code}) vs CSI500 - IR={row['information_ratio']:.2f}")
    ax.set_ylabel("Cumulative Return (indexed to 100)")
    ax.legend()

plt.tight_layout()
plt.savefig("active_quant_top5_vs_benchmark.png", dpi=150)
plt.show()


## Category comparison: active-quant vs. index-enhanced, both vs. CSI500

Equal-weighted, daily-rebalanced average of each category vs. the shared 000905.SH benchmark - the direct visual for whether one category tracked or diverged from the index more than the other over the same year.

In [ ]:
def group_average_cumulative_return(group_name):
    """Cumulative return of a hypothetical daily-rebalanced equal-weight portfolio of the group."""
    daily_returns = daily_returns_by_group[group_name]
    returns = daily_returns.drop(columns=["fund_name"]).astype(float) / 100
    avg_daily_return = returns.mean(axis=0)
    return (1 + avg_daily_return).cumprod() * 100


def benchmark_cumulative_return(group_name):
    benchmark_returns = benchmark_returns_by_group[group_name].copy()
    benchmark_returns.index = [pd.Timestamp(idx).strftime("%Y-%m-%d") for idx in benchmark_returns.index]
    bench_ret = benchmark_returns.iloc[:, 0] / 100
    return (1 + bench_ret).cumprod() * 100


active_quant_avg = group_average_cumulative_return("SH500_ActiveQuant")
index_enhanced_avg = group_average_cumulative_return("SH500")
csi500_cum = benchmark_cumulative_return("SH500")  # same 000905.SH index both groups benchmark against

fig, ax = plt.subplots(figsize=(11, 6))
x = pd.to_datetime(csi500_cum.index, format="%Y-%m-%d")

ax.plot(x, csi500_cum.values, label="CSI500 benchmark", color="gray", linestyle="--")
ax.plot(x, index_enhanced_avg.reindex(csi500_cum.index).values, label="Index-enhanced SH500 (equal-weight avg)")
ax.plot(x, active_quant_avg.reindex(csi500_cum.index).values, label="Active-quant SH500 (equal-weight avg)")

ax.set_ylabel("Cumulative Return (indexed to 100)")
ax.set_title("Category comparison vs CSI500: index-enhanced vs active-quant")
ax.legend()
plt.tight_layout()
plt.savefig("category_comparison_vs_csi500.png", dpi=150)
plt.show()
